# 02 -- Probing Analysis

Run linear probes on cached activations. The probing workstream asks: is information
about processing mode (S1-like vs S2-like), correctness, or bias susceptibility
linearly decodable from residual stream activations across layers?

**What this notebook does:**
1. Load activations from HDF5 and build probe targets
2. Run per-layer probes (mass-mean + logistic regression)
3. Plot layer-wise accuracy curves with Hewitt-Liang selectivity
4. Compare standard vs reasoning model probe curves
5. Cross-target comparison (task_type vs correctness vs bias_susceptible)
6. Self-correction trajectory analysis (reasoning models, T-positions)

**Modes:**
- **Synthetic** (default): runs on synthetic HDF5 with planted signal at layer 2
- **Real**: switch `DATA_PATH` to point to real activations

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

%matplotlib inline

REPO_ROOT = Path.cwd().parent
if str(REPO_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(REPO_ROOT / "src"))

from s1s2.utils.io import (
    open_activations, get_residual, load_problem_metadata,
    list_models, model_metadata, position_labels, position_valid,
)
from s1s2.probes import (
    ProbeRunner, RunnerConfig, build_target,
    MassMeanProbe, LogisticRegressionProbe, ALL_TARGETS,
)
from s1s2.viz.theme import set_paper_theme, COLORS, get_model_color

set_paper_theme()

In [ ]:
# ---- Mode selection ----
USE_SYNTHETIC = True  # Set to False for real data

if USE_SYNTHETIC:
    from tests.conftest import build_synthetic_hdf5
    DATA_PATH = REPO_ROOT / "data" / "activations" / "synthetic_nb02.h5"
    build_synthetic_hdf5(DATA_PATH)
    print(f"Built synthetic HDF5 at {DATA_PATH}")
    print("NOTE: Synthetic data has a planted signal at layer 2, dim 0.")
    print("Expect probe accuracy to peak at layer 2 for task_type target.")
else:
    DATA_PATH = REPO_ROOT / "data" / "activations" / "main.h5"  # REQUIRES REAL DATA
    print(f"Using real data at {DATA_PATH}")

In [ ]:
# --- Inspect the dataset ---
with open_activations(DATA_PATH) as f:
    meta = load_problem_metadata(f)
    models = list_models(f)
    print(f"Models: {models}")
    print(f"Problems: {len(meta['id'])}")
    for m in models:
        mm = model_metadata(f, m)
        labels = position_labels(f, m)
        print(f"  {m}: {mm.get('n_layers')} layers, "
              f"hidden={mm.get('hidden_dim')}, positions={labels}, "
              f"reasoning={mm.get('is_reasoning_model')}")

## Per-Layer Probing: Single Model

Run mass-mean and logistic regression probes at every layer for the `task_type` target
(conflict vs no-conflict). This is the headline result.

In [ ]:
# Configure a lightweight runner for notebook use
config = RunnerConfig(
    probes=("mass_mean", "logistic"),
    n_folds=5,
    n_seeds=1,             # 1 seed for speed; use 3+ for publication
    control_enabled=True,  # Hewitt-Liang control task
    control_n_shuffles=1,  # 1 for speed; use 3+ for publication
    n_permutations=100,    # 100 for speed; use 1000 for publication
    n_bootstrap=100,
    run_loco=False,        # Skip LOCO for speed
    seed=42,
)
runner = ProbeRunner(config)

model_key = models[0]
position = "P0"
target_name = "task_type"

print(f"Model: {model_key}")
print(f"Position: {position}")
print(f"Target: {target_name}")

In [ ]:
# Build the target labels
with open_activations(DATA_PATH) as f:
    td = build_target(f, model_key, target_name)
    mm = model_metadata(f, model_key)
    n_layers = int(mm["n_layers"])
    labels = position_labels(f, model_key)
    valid = position_valid(f, model_key)

print(f"Target: {td.target}, applicable: {td.mask.sum()}/{len(td.mask)}, "
      f"positive rate: {td.y.mean():.2f}")
print(f"Layers to probe: 0..{n_layers - 1}")

In [ ]:
# Run probes across all layers
# NOTE: ProbeRunner.run() expects FULL X (n_problems, hidden); it applies td.mask internally.
# ProbeResult.summary dict keys: roc_auc, roc_auc_mean, balanced_accuracy_mean,
#   selectivity, control_roc_auc, permutation_p, etc.
layer_results = []

for layer in range(n_layers):
    with open_activations(DATA_PATH) as f:
        X_full = get_residual(f, model_key, layer, position=position).astype(np.float32)

    result = runner.run(X=X_full, target_data=td, model=model_key,
                        layer=layer, position=position)

    for probe_name, probe_res in result.probes.items():
        s = probe_res.summary
        layer_results.append({
            "layer": layer,
            "probe": probe_name,
            "auc": s.get("roc_auc", 0.5),
            "bal_accuracy": s.get("balanced_accuracy_mean", 0.5),
            "control_auc": s.get("control_roc_auc", 0.5),
            "selectivity": s.get("selectivity", 0.0),
            "perm_p": s.get("permutation_p", 1.0),
        })
    print(f"  Layer {layer:2d}: ", end="")
    for probe_name, probe_res in result.probes.items():
        print(f"{probe_name} AUC={probe_res.summary.get('roc_auc', 0.5):.3f} ", end="")
    print()

df_layers = pd.DataFrame(layer_results)
print(f"\nDone. {len(df_layers)} results.")

In [ ]:
# Plot layer-wise AUC curves with Hewitt-Liang selectivity
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for probe_name, color in [("mass_mean", COLORS["standard"]),
                           ("logistic", COLORS["reasoning"])]:
    sub = df_layers[df_layers["probe"] == probe_name]
    # Left: AUC + control AUC
    axes[0].plot(sub["layer"], sub["auc"], "o-", label=f"{probe_name} (real)",
                 color=color, markersize=4)
    axes[0].plot(sub["layer"], sub["control_auc"], "x--",
                 label=f"{probe_name} (control)", color=color, alpha=0.5, markersize=4)
    # Right: selectivity (real AUC - control AUC)
    axes[1].plot(sub["layer"], sub["selectivity"], "o-", label=probe_name,
                 color=color, markersize=4)

axes[0].set_xlabel("Layer")
axes[0].set_ylabel("ROC-AUC")
axes[0].set_title(f"Probe AUC: {target_name} @ {position}")
axes[0].axhline(0.5, color="gray", linestyle=":", alpha=0.5, label="chance")
axes[0].legend(fontsize=8)
axes[0].set_ylim(0.3, 1.0)

axes[1].set_xlabel("Layer")
axes[1].set_ylabel("Selectivity (real AUC - control AUC)")
axes[1].set_title("Hewitt-Liang selectivity")
axes[1].axhline(0.05, color="red", linestyle=":", alpha=0.5,
                label="5pp threshold")
axes[1].axhline(0.0, color="gray", linestyle=":", alpha=0.3)
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

## Cross-Target Comparison

Compare probe curves across the four targets. If `task_type` and `processing_mode`
show similar curves, the probe is picking up genuine S1/S2 structure rather than
task-specific confounds.

In [ ]:
# Probe across targets with logistic regression at each layer
target_colors = {
    "task_type": COLORS["s1"],
    "correctness": COLORS["s2"],
    "bias_susceptible": COLORS["reasoning"],
    "processing_mode": COLORS["significant"],
}

multi_target_results = []

with open_activations(DATA_PATH) as f:
    for tgt_name in ALL_TARGETS:
        try:
            tgt_data = build_target(f, model_key, tgt_name)
        except (KeyError, ValueError) as e:
            print(f"  Skipping {tgt_name}: {e}")
            continue

        if tgt_data.y.sum() < 3 or (1 - tgt_data.y).sum() < 3:
            print(f"  Skipping {tgt_name}: too few positives/negatives")
            continue

        for layer in range(n_layers):
            X_full = get_residual(f, model_key, layer, position=position).astype(np.float32)
            result = runner.run(X=X_full, target_data=tgt_data, model=model_key,
                                layer=layer, position=position)
            for pname, pres in result.probes.items():
                if pname == "logistic":
                    s = pres.summary
                    multi_target_results.append({
                        "target": tgt_name,
                        "layer": layer,
                        "auc": s.get("roc_auc", 0.5),
                        "selectivity": s.get("selectivity", 0.0),
                    })
        print(f"  Done: {tgt_name}")

df_multi = pd.DataFrame(multi_target_results)

In [ ]:
if not df_multi.empty:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    for tgt_name in df_multi["target"].unique():
        sub = df_multi[df_multi["target"] == tgt_name]
        color = target_colors.get(tgt_name, "gray")
        axes[0].plot(sub["layer"], sub["auc"], "o-", label=tgt_name,
                     color=color, markersize=4)
        axes[1].plot(sub["layer"], sub["selectivity"], "o-", label=tgt_name,
                     color=color, markersize=4)

    axes[0].set_xlabel("Layer")
    axes[0].set_ylabel("ROC-AUC (logistic)")
    axes[0].set_title(f"Cross-target probe curves @ {position}")
    axes[0].axhline(0.5, color="gray", linestyle=":", alpha=0.5)
    axes[0].legend(fontsize=8)

    axes[1].set_xlabel("Layer")
    axes[1].set_ylabel("Selectivity")
    axes[1].set_title("Hewitt-Liang selectivity by target")
    axes[1].axhline(0.05, color="red", linestyle=":", alpha=0.5)
    axes[1].legend(fontsize=8)

    plt.tight_layout()
    plt.show()
else:
    print("No multi-target results to plot.")

## Self-Correction Trajectory (Reasoning Models)

For reasoning models, compare probe accuracy at different generation positions:
T0 (start of thinking) through Tend (end of thinking) to P2 (final answer).
If the model genuinely deliberates, we expect probe accuracy to increase from T0 to Tend.

In [ ]:
# Check which positions are valid for this model
with open_activations(DATA_PATH) as f:
    pos_labels = position_labels(f, model_key)
    pos_valid = position_valid(f, model_key)

# Only run trajectory analysis if T-positions are available
trajectory_positions = [p for p in ["P0", "T0", "T25", "T50", "T75", "Tend", "P2"]
                        if p in pos_labels]
# Filter to positions that are actually valid
valid_positions = []
for p in trajectory_positions:
    idx = pos_labels.index(p)
    if pos_valid[:, idx].any():
        valid_positions.append(p)

print(f"Available positions: {pos_labels}")
print(f"Valid for trajectory: {valid_positions}")

if len(valid_positions) <= 2:
    print("\nOnly P0/P2 valid (non-reasoning model in synthetic mode).")
    print("With real reasoning model data, T-position trajectory will appear here.")
else:
    # Pick a representative layer (mid-network)
    probe_layer = n_layers // 2
    trajectory_results = []

    with open_activations(DATA_PATH) as f:
        td = build_target(f, model_key, "task_type")
        for pos in valid_positions:
            X_full = get_residual(f, model_key, probe_layer, position=pos).astype(np.float32)
            result = runner.run(X=X_full, target_data=td, model=model_key,
                                layer=probe_layer, position=pos)
            for pname, pres in result.probes.items():
                if pname == "logistic":
                    s = pres.summary
                    trajectory_results.append({
                        "position": pos,
                        "auc": s.get("roc_auc", 0.5),
                        "selectivity": s.get("selectivity", 0.0),
                    })

    df_traj = pd.DataFrame(trajectory_results)

    fig, ax = plt.subplots(figsize=(8, 4))
    ax.plot(range(len(df_traj)), df_traj["auc"], "o-",
            color=COLORS["reasoning"], markersize=6, label="AUC")
    ax.fill_between(range(len(df_traj)),
                    df_traj["auc"] - 0.05,
                    df_traj["auc"] + 0.05,
                    alpha=0.2, color=COLORS["reasoning"])
    ax.set_xticks(range(len(df_traj)))
    ax.set_xticklabels(df_traj["position"])
    ax.set_xlabel("Generation position")
    ax.set_ylabel("Logistic probe AUC")
    ax.set_title(f"Self-correction trajectory (layer {probe_layer})")
    ax.axhline(0.5, color="gray", linestyle=":", alpha=0.5)
    ax.legend()
    plt.tight_layout()
    plt.show()

## Summary

**Key questions answered by this notebook:**

1. **Layer localization**: At which layer(s) does the S1/S2 distinction become linearly decodable? The peak layer is the candidate for downstream SAE/geometry analysis.
2. **Hewitt-Liang selectivity**: Is the signal genuine (selectivity > 5pp) or an artifact of probe expressiveness?
3. **Cross-target consistency**: Do `task_type` and `processing_mode` probe curves agree? Disagreement suggests the probe is learning task-specific rather than processing-mode features.
4. **Self-correction**: Does probe accuracy increase through the thinking trace? If accuracy is constant T0-Tend, reasoning is "performative" (decided at T0).

**Next steps:** Identify the peak layer(s) and feed into SAE feature analysis (notebook 03) and geometry (notebook 05).